# 🔧 Notebook 02 – Data Preprocessing & Feature Engineering
**Project:** Student Performance Prediction Using ML
**Intern:** SEERAPU SRAVANI | 24U45A4219 | CSE-AIML, JNTUGV

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

## 1. Load Raw Data

In [ ]:
df = pd.read_csv('../data/student_data.csv')
print('Raw shape:', df.shape)
df.head()

## 2. Handle Missing Values

In [ ]:
print('Missing before:', df.isnull().sum().sum())
# Fill numeric cols with mean (none expected in synthetic data)
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())
# Fill categorical with mode
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])
print('Missing after:', df.isnull().sum().sum())

## 3. Remove Duplicates

In [ ]:
print('Before dedup:', len(df))
df = df.drop_duplicates()
print('After dedup:', len(df))

## 4. Feature Engineering – Encoding Categoricals

In [ ]:
df_proc = df.drop(columns=['StudentID', 'FinalMarks', 'FinalGrade'])

le = LabelEncoder()
for col in ['Gender', 'SocioeconomicStatus', 'ParentEducation']:
    df_proc[col] = le.fit_transform(df_proc[col])
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Encode target
df_proc['Result'] = le.fit_transform(df_proc['Result'])
print('\nProcessed columns:', df_proc.columns.tolist())
df_proc.head()

## 5. Feature Scaling – StandardScaler

In [ ]:
X = df_proc.drop(columns=['Result'])
y = df_proc['Result']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
print('Feature means (should be ~0):\n', X_scaled_df.mean().round(3))
print('\nFeature stds (should be ~1):\n', X_scaled_df.std().round(3))

## 6. Train-Test Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]} samples')
print(f'Test : {X_test.shape[0]} samples')
print(f'\nClass balance (train):\n{pd.Series(y_train).value_counts()}')
print(f'\nClass balance (test):\n{pd.Series(y_test).value_counts()}')

## 7. Save Cleaned Dataset

In [ ]:
df_proc.to_csv('../data/student_data_cleaned.csv', index=False)
print('Cleaned dataset saved to data/student_data_cleaned.csv')

## ✅ Preprocessing Complete
- Missing values: handled via mean/mode imputation
- Categoricals: Label Encoded (Gender, SocioeconomicStatus, ParentEducation)
- Scaling: StandardScaler applied to all numeric features
- Split: 80% train, 20% test with stratification